# 🔬 Myopia Patient Clustering — Unsupervised ML
**Data Challenge | Code 213 — Data Science Bootcamp**

This notebook walks through:
- **Part 1:** Prepare the Data
- **Part 2:** Apply Dimensionality Reduction (PCA + t-SNE)
- **Part 3:** Perform Cluster Analysis with K-Means
- **Part 4:** Make a Recommendation

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

---
## Part 1: Prepare the Data

In [ ]:
# 1. Read the dataset
df = pd.read_csv('myopia.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
# Basic info
df.info()

In [ ]:
df.describe()

In [ ]:
# 2. Remove the MYOPIC target column
df_ml = df.drop(columns=['MYOPIC'])
print(f'Features after dropping MYOPIC: {df_ml.shape[1]}')
print(f'Columns: {list(df_ml.columns)}')

In [ ]:
# 3. Standardize the dataset
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_ml)

print(f'Mean after scaling : {X_scaled.mean():.6f}  (≈ 0)')
print(f'Std  after scaling : {X_scaled.std():.6f}  (≈ 1)')

---
## Part 2: Apply Dimensionality Reduction
### 2a — PCA

In [ ]:
# Preserve 90% of explained variance
pca = PCA(n_components=0.90, random_state=42)
X_pca = pca.fit_transform(X_scaled)

print(f'Original features  : {df_ml.shape[1]}')
print(f'PCA components     : {X_pca.shape[1]}  (preserving 90% variance)')
print(f'Explained variance : {np.round(pca.explained_variance_ratio_, 3)}')
print(f'Total variance kept: {pca.explained_variance_ratio_.sum()*100:.1f}%')

In [ ]:
# Plot PCA explained variance
fig, ax = plt.subplots(figsize=(8, 4))
components = range(1, X_pca.shape[1] + 1)

ax.bar(components, pca.explained_variance_ratio_ * 100,
       color='steelblue', edgecolor='white', label='Individual')
ax.plot(components, np.cumsum(pca.explained_variance_ratio_) * 100,
        'o--', color='tomato', linewidth=2, markersize=6, label='Cumulative')
ax.axhline(90, color='gold', linestyle='--', linewidth=1.5, label='90% threshold')

ax.set_xlabel('Principal Component')
ax.set_ylabel('Explained Variance (%)')
ax.set_title('PCA — Explained Variance per Component')
ax.legend()
plt.tight_layout()
plt.show()

print(f'\n→ PCA reduced the dataset from {df_ml.shape[1]} features to {X_pca.shape[1]} components.')

### 2b — t-SNE

In [ ]:
# Run t-SNE on PCA output
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
X_tsne = tsne.fit_transform(X_pca)
print(f't-SNE output shape: {X_tsne.shape}')

In [ ]:
# Scatter plot of t-SNE output
fig, ax = plt.subplots(figsize=(7, 6))

ax.scatter(X_tsne[:, 0], X_tsne[:, 1],
           c='steelblue', alpha=0.6, s=20, edgecolors='none')

ax.set_title('t-SNE Projection of Myopia Data')
ax.set_xlabel('t-SNE Dimension 1')
ax.set_ylabel('t-SNE Dimension 2')
plt.tight_layout()
plt.show()

print('→ The t-SNE plot shows some loose groupings, suggesting natural clusters exist in the data.')

---
## Part 3: Perform a Cluster Analysis with K-Means

In [ ]:
# Elbow method — inertia for k = 1 to 10
inertias = []
k_range = range(1, 11)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_pca)
    inertias.append(km.inertia_)
    print(f'  k={k:2d}  |  inertia = {km.inertia_:.2f}')

In [ ]:
# Plot the elbow curve
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(list(k_range), inertias, 'o-', color='steelblue',
        linewidth=2.5, markersize=8, markerfacecolor='white', markeredgewidth=2)

# Detect elbow automatically
deltas = np.diff(inertias)
delta2 = np.diff(deltas)
best_k = int(np.argmax(delta2) + 2)

ax.axvline(best_k, color='tomato', linestyle='--', linewidth=2,
           label=f'Elbow at k = {best_k}')
ax.scatter([best_k], [inertias[best_k - 1]], color='tomato', s=120, zorder=5)

ax.set_xlabel('Number of Clusters (k)', fontsize=12)
ax.set_ylabel('Inertia (Within-Cluster SSE)', fontsize=12)
ax.set_title('Elbow Plot — K-Means Clustering', fontsize=14, fontweight='bold')
ax.set_xticks(list(k_range))
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'→ The elbow appears at k = {best_k}')

In [ ]:
# Final K-Means with best k
km_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
labels = km_final.fit_predict(X_pca)

# Count patients per cluster
unique, counts = np.unique(labels, return_counts=True)
for cl, cnt in zip(unique, counts):
    print(f'  Cluster {cl + 1}: {cnt} patients')

In [ ]:
# Visualise clusters on t-SNE plot
palette = ['steelblue', 'tomato', 'gold', 'mediumseagreen', 'orchid']

fig, ax = plt.subplots(figsize=(8, 6))

for i in range(best_k):
    mask = labels == i
    ax.scatter(X_tsne[mask, 0], X_tsne[mask, 1],
               c=palette[i], alpha=0.75, s=22,
               edgecolors='none', label=f'Cluster {i + 1} (n={mask.sum()})')

ax.set_title(f't-SNE — Coloured by {best_k} K-Means Clusters', fontsize=13, fontweight='bold')
ax.set_xlabel('t-SNE Dimension 1')
ax.set_ylabel('t-SNE Dimension 2')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

---
## Part 4: Recommendation

> **Yes, the myopia patients can be clustered into 3 distinct groups.**  
> The elbow plot shows a clear inflection point at **k = 3**, and the t-SNE visualisation confirms that these three clusters correspond to meaningful, separable patient groupings — likely representing low-risk, moderate-risk, and high-risk myopia profiles — which could guide targeted clinical interventions.